# M3 · v0.1 · Tu primer programa con un LLM

Escribís tu **v0.1**: un programa que le pregunta a Gemini sobre **tu tema**, guarda respuesta, tokens y tiempo, y no se cae si algo falla.
Después comparás dos configuraciones, **una cosa por vez**, y estimás el costo.

**La tarea:** escribís lo mismo que en la práctica, en el mismo orden: 1. **las perillas** · 2. **`responder()`** · 3. **el bucle y el log** ·
4. **comparás** dos configuraciones y un error a propósito.

- Cada paso tiene el fragmento en una celda de texto: **escribilo** en la celda de código vacía de abajo (copiar y pegar también vale, pero cada renglón lo tenés que poder explicar).
- Después de cada paso hay un **chequeo**: si dice «OK», seguí.
- Corré las celdas en orden con **Shift + Enter**. La clave: **Secretos** (la llave de la izquierda) → `GOOGLE_API_KEY`.
- **Con tu asistente de IA** (el que usás, o Gemini dentro de Colab): pegale el fragmento y pedile «explicame cada renglón». Si sale un error, pegáselo. Cada renglón lo tenés que poder explicar.

<details><summary><b>¿Sale un error?</b></summary>

- **«No pude leer el secreto GOOGLE_API_KEY»** o **`API key not valid`:** revisá el nombre del secreto (exacto, en mayúsculas),
  que «Acceso del notebook» esté activado y que la clave esté bien copiada.
- **`IndentationError`:** dentro de `def`, `for`, `try` y `with`, cada renglón lleva 4 espacios más que el de arriba.
- **`503 UNAVAILABLE`** («high demand»): el modelo está saturado, no es tu código. Esperá un minuto y volvé a correr.
- **`429`** (*quota*, *RESOURCE_EXHAUSTED*): se terminó la cuota gratis del día o del minuto. Esperá o seguí mañana.
- **`NameError`:** salteaste una celda o Colab se reinició. Menú *Entorno de ejecución* → *Run before*.
- **Otro mensaje:** copialo tal cual en el foro «Consultas» del aula virtual.
</details>

## 1 · Preparar (una vez, unos 30 s)

In [ ]:
%pip install -q langchain==1.4.6 langchain-google-genai==4.4.2
import os
import re

try:
    from google.colab import userdata                      # en Colab: la clave sale de los Secretos
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass                                                   # fuera de Colab: GOOGLE_API_KEY ya tiene que estar
except Exception as error:                                 # no hay secreto con ese nombre o le falta el acceso
    print("No pude leer el secreto GOOGLE_API_KEY:", type(error).__name__)

from langchain.chat_models import init_chat_model

# Lo que en tu repo hace llm.py (acá no hay config.toml): el modelo y los errores en castellano
MODELO = "google_genai:gemini-3.5-flash-lite"


def nombre_modelo():
    return MODELO


def cargar_modelo():
    return init_chat_model(MODELO, max_output_tokens=300)  # tope de tokens de salida: cuida tu cuota


def explicar(error):
    texto = str(error)
    if "429" in texto or "RESOURCE_EXHAUSTED" in texto:
        return "429: se terminó la cuota del minuto o del día. Esperá, o cambiá MODELO."
    if "404" in texto or "NOT_FOUND" in texto:
        return "404: ese modelo no existe con ese nombre. Reintentar no sirve."
    if "503" in texto or "high demand" in texto:
        return "503: el modelo está saturado, no es tu código. Esperá un minuto."
    return None


def sin_claves(texto):
    return re.sub(r"AIza[0-9A-Za-z_\-]{20,}", "[clave tapada]", texto)  # el detalle del error va al log


print("Listo." if os.environ.get("GOOGLE_API_KEY") else "Falta la clave: Secretos → GOOGLE_API_KEY.")

## 2 · Las perillas

**Tu tema:** en `SISTEMA`, de qué se ocupa tu asistente y, si querés, algunos datos pegados (horarios, fechas, precios); en `PREGUNTAS`, **5 de tu tema** que se respondan con eso, cada una entre comillas y separadas con coma. Todavía es un solo pedido: lo que no le diste, no lo sabe.
En tu repo, además, va `from llm import cargar_modelo, explicar, nombre_modelo, sin_claves`: acá ya están en la preparación.

```python
import json, os, time
SISTEMA = "Sos un asistente para estudiantes. Respondé en español, en dos frases como máximo."
PREGUNTAS = ["¿Qué es un token en un LLM?", "¿Qué es una alucinación en un LLM?"]
modelo = cargar_modelo()  # el de MODELO (en tu repo, el de config.toml)
```

In [ ]:
# Escribí acá tus perillas: SISTEMA, PREGUNTAS (5 de tu tema) y modelo = cargar_modelo()

In [ ]:
assert isinstance(PREGUNTAS, list) and len(PREGUNTAS) == 5, f"PREGUNTAS tiene {len(PREGUNTAS)}: van 5 de tu tema"
assert all(isinstance(p, str) and p.strip() for p in PREGUNTAS), "Hay una pregunta vacía en PREGUNTAS"
assert SISTEMA.strip(), "SISTEMA está vacío: escribí el rol y las reglas"
print("Perillas OK:", len(PREGUNTAS), "preguntas con", nombre_modelo())

## 3 · `responder()`

Manda la pregunta, muestra la respuesta de a pedazos (*streaming*), los junta y mide.

```python
def responder(pregunta: str) -> dict:
    inicio, junta = time.perf_counter(), None
    for pedazo in modelo.stream([("system", SISTEMA), ("human", pregunta)]):
        print(pedazo.text, end="", flush=True)  # la respuesta aparece de a poco
        junta = pedazo if junta is None else junta + pedazo  # y se va juntando
    print()
    return {"respuesta": junta.text, "latencia_s": round(time.perf_counter() - inicio, 2),
            "tokens_entrada": junta.usage_metadata["input_tokens"], "tokens_salida": junta.usage_metadata["output_tokens"]}
```

In [ ]:
# Escribí acá responder(pregunta)

In [ ]:
r = responder(PREGUNTAS[0])
assert isinstance(r, dict), "responder() tiene que devolver un dict: ¿falta el return?"
faltan = {"respuesta", "tokens_entrada", "tokens_salida", "latencia_s"} - set(r)
assert not faltan, f"Al dict le falta: {faltan}"
print(r)

**Qué deberías ver:** la respuesta, de a poco, y abajo algo como `{'respuesta': '…', 'latencia_s': 1.2, 'tokens_entrada': 31, 'tokens_salida': 40}`.

## 4 · Las 5 y el log

Cada pregunta tiene un reintento si falla, y su resultado (o su error) queda en un renglón de `logs/v01.jsonl` (en Colab, en la carpeta de la izquierda).

```python
os.makedirs("logs", exist_ok=True)  # la primera vez no existe
for pregunta in PREGUNTAS:
    print("\n>", pregunta)
    for intento in (1, 2):  # el primero y un reintento
        try:
            fila = responder(pregunta)
            break
        except Exception as error:  # cuota, red, un modelo que no existe…
            fila = {"error": type(error).__name__, "detalle": sin_claves(str(error))[:200]}
            print(f"  Error (intento {intento} de 2):", explicar(error) or fila["error"])
            time.sleep(2)  # un 429 al instante vuelve a fallar
    with open("logs/v01.jsonl", "a", encoding="utf-8") as log:  # "a": agrega, no borra
        log.write(json.dumps({"modelo": nombre_modelo(), "sistema": SISTEMA, "pregunta": pregunta, **fila},
                             ensure_ascii=False) + "\n")
```

In [ ]:
# Escribí acá el bucle de las 5 preguntas

In [ ]:
import pandas as pd

assert os.path.exists("logs/v01.jsonl"), "No hay logs/v01.jsonl: ¿corriste la celda del bucle?"
log = pd.read_json("logs/v01.jsonl", lines=True)
con_tokens = log["tokens_salida"].notna().sum() if "tokens_salida" in log else 0
assert con_tokens >= 5, f"El log tiene {con_tokens} respuestas con tokens: tendrían que ser al menos 5"
print(f"Log OK: {len(log)} renglones, {con_tokens} con tokens.")
log.tail(5)

**Qué deberías ver:** las 5 respuestas, cada una debajo de su `>`, y el chequeo con los últimos renglones del log. El log es una tabla, una fila por pregunta, con las columnas `modelo`, `tokens_entrada`, `tokens_salida` y `latencia_s`.

**Primer control · 10–15 min:** el log tiene modelo, tokens y latencia de tus 5 preguntas.

## 5 · Comparar dos configuraciones

1. **Predecí** en `PREDICCION`: ¿cuál responde mejor? ¿Cuál gasta más?
2. Corré la celda que borra el log y después tu bucle: es la configuración **A**.
3. Cambiá **una** cosa: `SISTEMA` con reglas («si no estás seguro, decí "no sé"») en tu celda de perillas, u otro modelo en la celda de abajo.
4. Corré de nuevo tu celda de perillas y tu bucle: es la **B**. Quedan 10 respuestas.

In [ ]:
PREDICCION = ""  # antes de correr B: ¿cuál responde mejor? ¿Cuál gasta más? Por ejemplo: "el SISTEMA con reglas gasta más"

In [ ]:
if os.path.exists("logs/v01.jsonl"):  # empezar de cero: así la tabla tiene solo A y B
    os.remove("logs/v01.jsonl")
print("Log borrado: corré A.")

In [ ]:
# Si comparás modelos, escribí acá MODELO = "google_genai:gemini-3.1-flash-lite" y corré la celda

## 6 · Un error a propósito

1. En la celda de arriba poné un modelo que no existe: `MODELO = "google_genai:gemini-que-no-existe"`.
2. **Predecí** qué queda en el log. Corré tu celda de perillas y tu bucle.
3. ¿Sirvió el reintento? Mirá los últimos renglones con la celda de abajo.
4. Volvé al modelo de antes (`MODELO = "google_genai:gemini-3.5-flash-lite"`) y corré tu celda de perillas.

In [ ]:
print("".join(open("logs/v01.jsonl", encoding="utf-8").readlines()[-5:]))  # los últimos 5 renglones, tal cual

## 7 · La tabla

Una fila por configuración: tokens promedio y latencia mediana. Es un `GROUP BY modelo` con promedios y mediana, como «Agrupar por» en Power BI. Si variaste el `SISTEMA`, cambiá `"modelo"` por `"sistema"`.

```python
import pandas as pd
grupos = pd.read_json("logs/v01.jsonl", lines=True).groupby("modelo")  # los errores no tienen tokens: no cuentan
print(grupos.agg({"tokens_entrada": "mean", "tokens_salida": "mean", "latencia_s": "median"}).round(1).to_string())
```

In [ ]:
# Escribí acá la tabla

**Qué deberías ver:** una fila por configuración. El modelo que no existe sale con `NaN`: sin respuestas no hay tokens.

## 8 · Costo por mes

A 1000 preguntas por día, con tus tokens promedio y el precio oficial **con fecha**
([Gemini](https://ai.google.dev/gemini-api/docs/pricing), [OpenAI](https://developers.openai.com/api/docs/pricing); *Paid tier*), en dólares por millón de tokens:

costo por pregunta = tokens de entrada × precio de entrada / 1.000.000 + tokens de salida × precio de salida / 1.000.000

costo por mes = costo por pregunta × 1000 × 30

Sale en una planilla, una celda por término. En Python: `1_000_000`, y `print(f"{costo:.7f}")` para ver los decimales.

In [ ]:
# Tu cuenta: costo por pregunta y costo por mes

## Entregar (entrega de avance 1, hasta el mar 3/11, 23:59)

- **Guardá este notebook en tu repo:** *Archivo → Guardar una copia en GitHub* → tu repositorio → `v01.ipynb`.
- **Evidencia** (una celda de texto al final, acá mismo): la tabla de las dos configuraciones, cuál elegiste y por qué,
  qué quedó en el log con el modelo que no existe (¿sirvió el reintento?) y el costo por mes.

**Microdefensa:** ¿qué error vale reintentar y cuál no? ¿Cómo los separarías en el `except`? Confianza del 1 al 4.